In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

train = pd.read_csv("/content/engineered_train.csv")
validation = pd.read_csv("/content/engineered_validation.csv")

In [ ]:
print("Train shape:", train.shape)
print("Validation shape:", validation.shape)

Train shape: (280741, 84)
Validation shape: (118108, 84)


In [ ]:
train.head()


,TransactionID,isFraud,ProductCD,card1,card2,card3,card4,card5,card6,addr1,...,card1_freq,card2_freq,addr1_freq,P_emaildomain_freq,DeviceInfo_freq,id_02_isna,id_05_isna,id_06_isna,id_11_isna,DeviceInfo_isna
0,2987000,0,0,13926.0,NaN,150.0,0.0,142.0,0.0,315.0,...,0.000089,0.000000,0.044032,0.000000,0.00000,1.0,1.0,1.0,1.0,1.0
1,2987001,0,0,2755.0,404.0,150.0,1.0,102.0,0.0,325.0,...,0.001207,0.005552,0.081355,0.458051,0.00000,1.0,1.0,1.0,1.0,1.0
2,2987002,0,0,4663.0,490.0,150.0,2.0,166.0,1.0,330.0,...,0.001875,0.064821,0.047355,0.010230,0.00000,1.0,1.0,1.0,1.0,1.0
3,2987003,0,0,18132.0,567.0,150.0,1.0,117.0,1.0,476.0,...,0.007161,0.010582,0.018274,0.202720,0.00000,1.0,1.0,1.0,1.0,1.0
4,2987004,0,1,4497.0,514.0,150.0,1.0,102.0,0.0,420.0,...,0.000028,0.025339,0.006814,0.458051,0.00009,0.0,1.0,1.0,0.0,0.0


In [ ]:
train['isFraud'].value_counts()

,count
isFraud,
0,271339
1,9402


In [ ]:
X_train = train.drop(columns=["isFraud"])
y_train = train["isFraud"]

X_val = validation.drop(columns=["isFraud"])
y_val = validation["isFraud"]

In [ ]:
X_train = X_train.drop(columns=["TransactionID", "uid"])
X_val = X_val.drop(columns=["TransactionID", "uid"])

In [ ]:
cat_cols = X_train.select_dtypes(include=["object"]).columns

print(cat_cols)

Index(['P_emaildomain_bin', 'R_emaildomain_bin', 'DeviceInfo_bin'], dtype='object')


In [ ]:
X_train = pd.get_dummies(
    X_train,
    columns=cat_cols,
    dummy_na=True
)

X_val = pd.get_dummies(
    X_val,
    columns=cat_cols,
    dummy_na=True
)

In [ ]:
X_train, X_val = X_train.align(
    X_val,
    join="left",
    axis=1,
    fill_value=0
)

In [ ]:
X_train = X_train.fillna(-999)
X_val = X_val.fillna(-999)

In [ ]:
print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)

print("Remaining missing values:",
      X_train.isna().sum().sum())

Training features: (280741, 131)
Validation features: (118108, 131)
Remaining missing values: 0


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

RandomForestClassifier was selected as the first model because:

It works well with tabular data.
It can capture non-linear relationships between transaction features.
It is an ensemble of multiple decision trees, making it more robust than a single tree.
It gives us a baseline to compare against more advanced boosting models such as LightGBM, XGBoost, and CatBoost.

In [ ]:
rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

Before using our more sophisticated boosting architecture, how well can a standard ensemble model solve this problem

In [ ]:
y_prob = rf_model.predict_proba(X_val)[:, 1]

print(y_prob[:10])

[0.015 0.005 0.065 0.19  0.    0.05  0.045 0.01  0.01  0.005]


In [ ]:
y_pred = (y_prob >= 0.5).astype(int)

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    average_precision_score
)

precision = precision_score(y_val, y_pred, zero_division=0)
recall = recall_score(y_val, y_pred, zero_division=0)
f1 = f1_score(y_val, y_pred, zero_division=0)
f2 = fbeta_score(y_val, y_pred, beta=2, zero_division=0)
roc_auc = roc_auc_score(y_val, y_prob)
pr_auc = average_precision_score(y_val, y_prob)

print("Random Forest Results")
print("---------------------")
print("Precision :", precision)
print("Recall    :", recall)
print("F1 Score  :", f1)
print("F2 Score  :", f2)
print("ROC-AUC   :", roc_auc)
print("PR-AUC    :", pr_auc)

Random Forest Results
---------------------
Precision : 0.8664122137404581
Recall    : 0.16756889763779528
F1 Score  : 0.2808247422680412
F2 Score  : 0.19980049289989438
ROC-AUC   : 0.830272344290655
PR-AUC    : 0.4249281518310169


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val, y_pred)

print(cm)

[[113939    105]
 [  3383    681]]


In [ ]:
tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives  :", tp)

True Negatives : 113939
False Positives: 105
False Negatives: 3383
True Positives  : 681


In [ ]:
rf_results = {
    "Model": "Random Forest",
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "F2": f2,
    "ROC-AUC": roc_auc,
    "PR-AUC": pr_auc,
    "Threshold": 0.5
}

print(rf_results)

{'Model': 'Random Forest', 'Precision': 0.8664122137404581, 'Recall': 0.16756889763779528, 'F1': 0.2808247422680412, 'F2': 0.19980049289989438, 'ROC-AUC': np.float64(0.830272344290655), 'PR-AUC': np.float64(0.4249281518310169), 'Threshold': 0.5}


The model is missing a large proportion of fraud cases when using the default 0.5 threshold. The default classification threshold of 0.5 is not appropriate for our fraud-detection objective.

**Threshold tuning**

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score

threshold_results = []

for threshold in np.arange(0.05, 0.51, 0.05):

    y_pred_threshold = (y_prob >= threshold).astype(int)

    precision = precision_score(
        y_val,
        y_pred_threshold,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred_threshold,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred_threshold,
        zero_division=0
    )

    f2 = fbeta_score(
        y_val,
        y_pred_threshold,
        beta=2,
        zero_division=0
    )

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "F2": f2
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,Threshold,Precision,Recall,F1,F2
0,0.05,0.114593,0.718996,0.197680,0.349898
1,0.10,0.306661,0.495079,0.378729,0.440899
2,0.15,0.534443,0.402805,0.459380,0.423676
3,0.20,0.632840,0.357530,0.456918,0.391602
4,0.25,0.713576,0.318159,0.440095,0.357815
5,0.30,0.766404,0.287402,0.418039,0.328459
6,0.35,0.805296,0.254429,0.386687,0.294755
7,0.40,0.828724,0.228593,0.358341,0.267307
8,0.45,0.842553,0.194882,0.316547,0.230286
9,0.50,0.866412,0.167569,0.280825,0.199800


In [ ]:
best_f2 = threshold_df.loc[
    threshold_df["F2"].idxmax()
]

print(best_f2)

Threshold    0.100000
Precision    0.306661
Recall       0.495079
F1           0.378729
F2           0.440899
Name: 1, dtype: float64


We trained Random Forest first not because it is necessarily our final model, but because it gives us a bagging baseline against which we can scientifically measure whether boosting models like LightGBM actually improve fraud detection.

It can rank fraud reasonably well, but at the default threshold it misses too much fraud, so we need to investigate whether LightGBM can give us a better precision-recall/F2 tradeoff.

In [ ]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print("Normal:", negative)
print("Fraud:", positive)
print("Scale Pos Weight:", scale_pos_weight)

Normal: 271339
Fraud: 9402
Scale Pos Weight: 28.859710699851096


**Final Conclusion**

Random Forest provides a useful baseline, but its low Recall shows that there is room for improvement. Therefore, we will next evaluate boosting algorithms such as LightGBM, XGBoost, and CatBoost to see whether they provide a better Precision–Recall and F2 trade-off.